In [2]:
pip install fiftyone

Defaulting to user installation because normal site-packages is not writeable
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of wsproto to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 24.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 20.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 27.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 28.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 29.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.9/935.9 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

In [3]:
import fiftyone.zoo as foz
dataset = foz.load_zoo_dataset("coco-2017", split="validation")
dataset.persistent = True

 100% |██████|    1.9Gb/1.9Gb [1.2m elapsed, 0s remaining, 36.9Mb/s]        
Extracting annotations to '/home/jupyter-st126222/fiftyone/coco-2017/raw/instances_val2017.json'
 100% |██████|    6.1Gb/6.1Gb [3.6m elapsed, 0s remaining, 30.6Mb/s]       
Extracting images to '/home/jupyter-st126222/fiftyone/coco-2017/validation/data'
Writing annotations to '/home/jupyter-st126222/fiftyone/coco-2017/validation/labels.json'
Dataset info written to '/home/jupyter-st126222/fiftyone/coco-2017/info.json'
Loading 'coco-2017' split 'validation'
 100% |███████████████| 5000/5000 [28.4s elapsed, 0s remaining, 169.4 samples/s]      
Dataset 'coco-2017-validation' created


In [4]:
import os
path2data = os.path.expanduser('~/fiftyone/coco-2017/validation/data')
path2json = os.path.expanduser('~/fiftyone/coco-2017/raw/instances_val2017.json')

print('Images exist :', os.path.isdir(path2data))
print('JSON exists  :', os.path.isfile(path2json))

Images exist : True
JSON exists  : True


In [1]:
import torch
import numpy as np
import os
os.chdir('/home/jupyter-st126222/A2')

from model import MyDarknet
from util import write_results, load_classes

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# load model with trained weights
model = MyDarknet('cfg/yolov4.cfg')
state_dict = torch.load('checkpoints/yolov4_iou_epoch5.pt', map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

# load test image
import cv2
img_bgr = cv2.imread('images/dog-cycle-car.png')
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_resized = cv2.resize(img_rgb, (608, 608))
img_tensor  = torch.from_numpy(img_resized).permute(2,0,1).float().unsqueeze(0) / 255.0
img_tensor  = img_tensor.to(device)

with torch.no_grad():
    pred = model(img_tensor, True)

print('Output shape    :', pred.shape)
print('Output min/max  :', pred.min().item(), pred.max().item())

# check confidence scores
conf = torch.sigmoid(pred[0, :, 4])
print('Conf min/max    :', conf.min().item(), conf.max().item())
print('Conf > 0.5      :', (conf > 0.5).sum().item())
print('Conf > 0.1      :', (conf > 0.1).sum().item())
print('Conf > 0.01     :', (conf > 0.01).sum().item())

# try write_results
detections = write_results(pred, 0.5, 80, nms_conf=0.4)
print('Detections (0.5):', detections)

detections2 = write_results(pred, 0.1, 80, nms_conf=0.4)
print('Detections (0.1):', detections2)

/tmp/ipykernel_3008113/3622396909.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('checkpoints/yolov4_iou_epoch5.pt', map_location=device)


Output shape    : torch.Size([1, 22743, 85])
Output min/max  : 6.9872565205801335e-15 607.08935546875
Conf min/max    : 0.5 0.7236137986183167
Conf > 0.5      : 20278
Conf > 0.1      : 22743
Conf > 0.01     : 22743
Detections (0.5): tensor([[0.0000e+00, 9.2328e+01, 1.1957e+02, 4.4874e+02, 4.4202e+02, 5.4577e-01,
         9.9887e-01, 1.0000e+00],
        [0.0000e+00, 3.7094e+02, 8.4415e+01, 5.4034e+02, 1.7783e+02, 9.6246e-01,
         9.4722e-01, 7.0000e+00],
        [0.0000e+00, 1.0398e+02, 2.3979e+02, 2.4819e+02, 5.6981e+02, 8.8636e-01,
         9.9795e-01, 1.6000e+01]], device='cuda:0')
Detections (0.1): tensor([[0.0000e+00, 9.2328e+01, 1.1957e+02, 4.4874e+02, 4.4202e+02, 5.4577e-01,
         9.9887e-01, 1.0000e+00],
        [0.0000e+00, 3.7094e+02, 8.4415e+01, 5.4034e+02, 1.7783e+02, 9.6246e-01,
         9.4722e-01, 7.0000e+00],
        [0.0000e+00, 1.0398e+02, 2.3979e+02, 2.4819e+02, 5.6981e+02, 8.8636e-01,
         9.9795e-01, 1.6000e+01],
        [0.0000e+00, 5.4009e+02, 1.1496e+

In [1]:
import torch
import numpy as np
import os
os.chdir('/home/jupyter-st126222/A2')

from model import MyDarknet
from dataset import CustomCoco
from train import collate_fn
from torch.utils.data import DataLoader, Subset
import albumentations as A

device = torch.device('cuda')

# load model
model = MyDarknet('cfg/yolov4.cfg')
state_dict = torch.load('checkpoints/yolov4_iou_epoch5.pt', map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

# load one batch from val
val_transform = A.Compose([
    A.Resize(608, 608),
], bbox_params=A.BboxParams(
    format='coco',
    label_fields=['category_ids'],
    min_visibility=0.1,
))

full_dataset = CustomCoco(
    root      = os.path.expanduser('~/fiftyone/coco-2017/validation/data'),
    annFile   = os.path.expanduser('~/fiftyone/coco-2017/raw/instances_val2017.json'),
    transform = val_transform,
    img_size  = 608,
    model_ver = 'v4',
)

val_dataset = Subset(full_dataset, list(range(4000, 4010)))
val_loader  = DataLoader(val_dataset, batch_size=2, shuffle=False,
                         num_workers=0, collate_fn=collate_fn)

# get one batch
inputs, labels, bboxes = next(iter(val_loader))

print('inputs type  :', type(inputs))
print('inputs[0] type:', type(inputs[0]))
print('inputs[0] shape:', np.array(inputs[0]).shape)

# convert inputs
inputs_t = torch.from_numpy(
    np.array(inputs)
).squeeze(1).permute(0,3,1,2).float().to(device) / 255.0
print('inputs_t shape:', inputs_t.shape)

# forward pass
with torch.no_grad():
    outputs = model(inputs_t, True)

print('outputs shape :', outputs.shape)
print('outputs min/max:', outputs.min().item(), outputs.max().item())

# check conf
conf = outputs[..., 4]
print('conf min/max  :', conf.min().item(), conf.max().item())
print('conf > 0.05   :', (conf > 0.05).sum().item())
print('conf > 0.5    :', (conf > 0.5).sum().item())

# check labels
label_i  = labels[0]
print('label shape   :', label_i.shape)
obj_mask = label_i[..., 4] > 0
print('num gt objects:', obj_mask.sum().item())
gt_xywh  = label_i[obj_mask][..., :4]
print('gt boxes      :', gt_xywh[:3])

/tmp/ipykernel_3012626/281091075.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('checkpoints/yolov4_iou_epoch5.pt', map_location=device)


loading annotations into memory...
Done (t=0.47s)
creating index...
index created!
inputs type  : <class 'tuple'>
inputs[0] type: <class 'numpy.ndarray'>
inputs[0] shape: (608, 608, 3)
inputs_t shape: torch.Size([2, 3, 608, 608])
outputs shape : torch.Size([2, 22743, 85])
outputs min/max: 1.0515263639362638e-31 39133.16796875
conf min/max  : 2.937954445155454e-14 0.9846813678741455
conf > 0.05   : 238
conf > 0.5    : 41
label shape   : torch.Size([22743, 85])
num gt objects: 5
gt boxes      : tensor([[2809.6060, 2510.1997, 2294.2119, 4668.1865],
        [ 891.4420, 2935.7715,  206.1880,  297.4218],
        [ 907.9340, 3246.2930, 1655.3560, 1180.4597]])


In [2]:
import torch
import numpy as np
import os
os.chdir('/home/jupyter-st126222/A2')

from model import MyDarknet
from dataset import CustomCoco
from train import collate_fn
from torch.utils.data import DataLoader, Subset
import albumentations as A

device = torch.device('cuda')

# load model
model = MyDarknet('cfg/yolov4.cfg')
state_dict = torch.load('checkpoints/yolov4_iou_epoch5.pt', map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

# load one batch
val_transform = A.Compose([
    A.Resize(608, 608),
], bbox_params=A.BboxParams(
    format='coco',
    label_fields=['category_ids'],
    min_visibility=0.1,
))

full_dataset = CustomCoco(
    root      = os.path.expanduser('~/fiftyone/coco-2017/validation/data'),
    annFile   = os.path.expanduser('~/fiftyone/coco-2017/raw/instances_val2017.json'),
    transform = val_transform,
    img_size  = 608,
    model_ver = 'v4',
)

val_dataset = Subset(full_dataset, list(range(4000, 4002)))
val_loader  = DataLoader(val_dataset, batch_size=2, shuffle=False,
                         num_workers=0, collate_fn=collate_fn)

inputs, labels, bboxes = next(iter(val_loader))

inputs_t = torch.from_numpy(
    np.array(inputs)
).squeeze(1).permute(0,3,1,2).float().to(device) / 255.0

with torch.no_grad():
    outputs = model(inputs_t, True)

# ── Check predictions ──────────────────────────────────────────────
print('=== PREDICTIONS ===')
conf        = outputs[..., 4]
cls_scores  = outputs[..., 5:]
cls_max, cls_ids = cls_scores.max(dim=-1)
scores      = conf * cls_max
scores      = scores.clamp(0, 1)

print('conf   min/max:', conf.min().item(), conf.max().item())
print('scores min/max:', scores.min().item(), scores.max().item())
print('scores > 0.05 :', (scores > 0.05).sum().item())
print('scores > 0.01 :', (scores > 0.01).sum().item())

# show top 5 predictions for image 0
top5 = scores[0].topk(5)
print('\nTop 5 scores image 0:', top5.values)
print('Top 5 cls   image 0:', cls_ids[0][top5.indices])
print('Top 5 boxes image 0:', outputs[0][top5.indices, :4])

# ── Check ground truth ─────────────────────────────────────────────
print('\n=== GROUND TRUTH ===')
label_i  = labels[0]
obj_mask = label_i[..., 4] > 0
gt_xywh  = label_i[obj_mask][..., :4]
gt_cls   = label_i[obj_mask][..., 5:].argmax(dim=-1)
print('num gt objects:', obj_mask.sum().item())
print('gt boxes      :', gt_xywh)
print('gt classes    :', gt_cls)

# ── Manually compute IoU between top pred and gt ───────────────────
print('\n=== MANUAL IoU CHECK ===')
if gt_xywh.shape[0] > 0 and (scores[0] > 0.05).sum() > 0:
    # convert pred xywh to xyxy
    top_idx  = scores[0].argmax()
    pred_box = outputs[0][top_idx, :4].cpu()
    px1 = pred_box[0] - pred_box[2]/2
    py1 = pred_box[1] - pred_box[3]/2
    px2 = pred_box[0] + pred_box[2]/2
    py2 = pred_box[1] + pred_box[3]/2
    print('Best pred box (xyxy):', px1.item(), py1.item(), px2.item(), py2.item())
    print('Best pred score     :', scores[0][top_idx].item())
    print('Best pred class     :', cls_ids[0][top_idx].item())

    # convert gt xywh to xyxy
    for i, gt in enumerate(gt_xywh):
        gx1 = gt[0] - gt[2]/2
        gy1 = gt[1] - gt[3]/2
        gx2 = gt[0] + gt[2]/2
        gy2 = gt[1] + gt[3]/2
        print(f'GT box {i} (xyxy)    :', gx1.item(), gy1.item(), gx2.item(), gy2.item())

/tmp/ipykernel_3012626/779542385.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('checkpoints/yolov4_iou_epoch5.pt', map_location=device)


loading annotations into memory...
Done (t=0.46s)
creating index...
index created!
=== PREDICTIONS ===
conf   min/max: 2.937954445155454e-14 0.9846813678741455
scores min/max: 2.4440646531495797e-15 0.9846276044845581
scores > 0.05 : 227
scores > 0.01 : 438

Top 5 scores image 0: tensor([0.9846, 0.9837, 0.9357, 0.9350, 0.9338], device='cuda:0')
Top 5 cls   image 0: tensor([38, 38, 32, 32, 32], device='cuda:0')
Top 5 boxes image 0: tensor([[112.9061, 400.0239, 203.3772, 145.6390],
        [112.9052, 399.9469, 201.7425, 147.3848],
        [112.9704, 366.1022,  25.6272,  40.0956],
        [110.9512, 366.1288,  25.6227,  40.2044],
        [112.9728, 366.1336,  25.0836,  40.3198]], device='cuda:0')

=== GROUND TRUTH ===
num gt objects: 5
gt boxes      : tensor([[2809.6060, 2510.1997, 2294.2119, 4668.1865],
        [ 891.4420, 2935.7715,  206.1880,  297.4218],
        [ 907.9340, 3246.2930, 1655.3560, 1180.4597],
        [ 907.9340, 3246.2930, 1655.3560, 1180.4597],
        [ 907.9340, 3246.